# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_token")

from huggingface_hub import login

login(token=HF_TOKEN)

In [2]:
from datasets import load_dataset

dim_content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train",
    token=HF_TOKEN
)

dim_content = dim_content.to_pandas()

print("dim_content:", dim_content.shape)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

dim_content: (519606, 26)


In [3]:
print("Relevant columns:")
print([
    "content_created_date",
    "content_updated_date",
    "last_optimized_date",
    "optimization_eligible_date",
    "search_volume"
])

Relevant columns:
['content_created_date', 'content_updated_date', 'last_optimized_date', 'optimization_eligible_date', 'search_volume']


In [4]:
dim_content[
    [
        "content_created_date",
        "content_updated_date",
        "last_optimized_date",
        "optimization_eligible_date",
        "search_volume"
    ]
].head()

,content_created_date,content_updated_date,last_optimized_date,optimization_eligible_date,search_volume
0,2026-05-30,2026-07-01,None,None,30.0
1,2026-06-12,2026-07-01,None,None,10.0
2,2026-05-09,2026-07-01,None,None,480.0
3,2026-06-15,2026-06-15,None,None,0.0
4,2026-05-21,2026-06-01,None,None,2400.0


In [5]:
print("Latest content_updated_date:",
      dim_content["content_updated_date"].max())

print("Latest content_created_date:",
      dim_content["content_created_date"].max())

print("Search volume summary:")
print(dim_content["search_volume"].describe())

Latest content_updated_date: 2026-07-06
Latest content_created_date: 2026-07-06
Search volume summary:
count    376984.000000
mean        209.574544
std        3207.498742
min           0.000000
25%           0.000000
50%          10.000000
75%          20.000000
max      368000.000000
Name: search_volume, dtype: float64


In [6]:
import pandas as pd

reference_date = pd.Timestamp("2026-07-06")

# Calculate content age in days
dim_content["content_age_days"] = (
    reference_date - pd.to_datetime(dim_content["content_updated_date"])
).dt.days

print("CONTENT AGE")
print(dim_content["content_age_days"].describe())

print("\nCONTENT AGE QUANTILES")
print(
    dim_content["content_age_days"]
    .quantile([0.25, 0.50, 0.75, 0.90, 0.95])
)

print("\nSEARCH VOLUME QUANTILES")
print(
    dim_content["search_volume"]
    .quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

CONTENT AGE
count    519606.000000
mean        131.148043
std         190.591422
min           0.000000
25%          35.000000
50%          47.000000
75%         131.000000
max         616.000000
Name: content_age_days, dtype: float64

CONTENT AGE QUANTILES
0.25     35.0
0.50     47.0
0.75    131.0
0.90    571.0
0.95    588.0
Name: content_age_days, dtype: float64

SEARCH VOLUME QUANTILES
0.25       0.0
0.50      10.0
0.75      20.0
0.90     110.0
0.95     390.0
0.99    3600.0
Name: search_volume, dtype: float64


In [7]:
# importing the second dataset
from huggingface_hub import hf_hub_download
fact_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

fact_daily = pd.read_parquet(fact_path)

print("fact_daily:", fact_daily.shape)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_daily: (9841378, 30)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

##Rule:
Rank content pages using a simple action score based on content staleness and search demand. Older content receives a higher score because it may need refreshing, while pages with higher search demand receive a higher score because they have more potential value. The score is used to prioritize pages for review; it is not a prediction of performance or a final recommendation.

###Reason codes:

1. STALE_HIGH_DEMAND: Content is old and has meaningful search demand.
2. STALE_LOW_DEMAND: Content is old but has little search demand.
3. FRESH_HIGH_DEMAND: Content is relatively fresh and has meaningful search demand.
4. FRESH_LOW_DEMAND: Content is relatively fresh and has little search demand.

In [8]:
# Signal 1: Content staleness

dim_content["staleness_bucket"] = pd.cut(
    dim_content["content_age_days"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["Fresh (0-30d)", "Recent (31-90d)", "Aging (91-180d)", "Stale (181+d)"]
)

staleness_table = (
    dim_content["staleness_bucket"]
    .value_counts(sort=False)
    .rename_axis("staleness_bucket")
    .reset_index(name="n")
)

print("SIGNAL 1: CONTENT STALENESS")
print(staleness_table)

SIGNAL 1: CONTENT STALENESS
  staleness_bucket       n
0    Fresh (0-30d)  121939
1  Recent (31-90d)  260753
2  Aging (91-180d)   36074
3    Stale (181+d)  100840


In [9]:
# step 1
print("fact_daily:", fact_daily.shape)
print("Earliest:", fact_daily["report_date"].min())
print("Latest:", fact_daily["report_date"].max())

fact_daily: (9841378, 30)
Earliest: 2026-03-01
Latest: 2026-03-31


In [10]:
# step 2
content_perf = (
    fact_daily
    .groupby("content_hash_id", as_index=False)
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_data_available=("gsc_data_available", "max")
    )
)

print("content_perf:", content_perf.shape)
content_perf.head()

content_perf: (331437, 4)


,content_hash_id,gsc_impressions,gsc_clicks,gsc_data_available
0,content_000005d4ced12088,86,0,True
1,content_00001e488b74b799,0,0,False
2,content_00007bd2985b77c3,47,0,True
3,content_00008950670cb6b5,0,0,False
4,content_0000a348850eb1fc,0,0,False


In [11]:
# step 3
signal_df = dim_content.merge(
    content_perf,
    on="content_hash_id",
    how="left"
)

print("signal_df:", signal_df.shape)
print(signal_df[
    [
        "content_hash_id",
        "content_age_days",
        "staleness_bucket",
        "search_volume",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_data_available"
    ]
].head())

signal_df: (519606, 31)
            content_hash_id  content_age_days staleness_bucket  search_volume  \
0  content_004de9653278b5a4                 5    Fresh (0-30d)           30.0   
1  content_00dc5efae381b2ab                 5    Fresh (0-30d)           10.0   
2  content_01410f2556c327ac                 5    Fresh (0-30d)          480.0   
3  content_019f27f634053ca7                21    Fresh (0-30d)            0.0   
4  content_01efa71faea45dcc                35  Recent (31-90d)         2400.0   

   gsc_impressions  gsc_clicks gsc_data_available  
0              NaN         NaN                NaN  
1              NaN         NaN                NaN  
2              NaN         NaN                NaN  
3              NaN         NaN                NaN  
4              NaN         NaN                NaN  


In [12]:
# step 4
staleness_check = (
    signal_df[
        signal_df["gsc_data_available"] == True
    ]
    .groupby("staleness_bucket")
    .agg(
        n=("content_hash_id", "nunique"),
        median_impressions=("gsc_impressions", "median"),
        mean_impressions=("gsc_impressions", "mean"),
        median_clicks=("gsc_clicks", "median"),
        total_impressions=("gsc_impressions", "sum")
    )
    .reset_index()
)

print(staleness_check)

  staleness_bucket      n  median_impressions  mean_impressions  \
0    Fresh (0-30d)  60938               805.0       3173.779760   
1  Recent (31-90d)  87827                46.0        600.174582   
2  Aging (91-180d)  26387               313.0       1290.350021   
3    Stale (181+d)   1586                 4.0        311.348676   

   median_clicks  total_impressions  
0            1.0        193403791.0  
1            0.0         52711533.0  
2            0.0         34048466.0  
3            0.0           493799.0  


/tmp/ipykernel_3507/954578494.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("staleness_bucket")


Verdict: MIXED<br>
Staleness shows a directional relationship with visibility at the extremes: Fresh content has a median of 805 impressions while Stale content has only 4. However, the middle buckets are not monotonic because Aging content (313) has higher median impressions than Recent content (46). Therefore, staleness is a useful directional signal but not a reliable standalone indicator of performance.

# Signal 2: Search Volume

In [13]:
# step 1 creating search volume brackets
signal_df["search_volume_bucket"] = pd.cut(
    signal_df["search_volume"],
    bins=[-1, 0, 20, 110, 390, float("inf")],
    labels=[
        "No demand (0)",
        "Low (1-20)",
        "Moderate (21-110)",
        "High (111-390)",
        "Very high (391+)"
    ]
)

print(
    signal_df["search_volume_bucket"]
    .value_counts()
    .sort_index()
)

search_volume_bucket
No demand (0)        163631
Low (1-20)           127670
Moderate (21-110)     50948
High (111-390)        17837
Very high (391+)      16898
Name: count, dtype: int64


In [14]:
# step 2 making the actual signal table
volume_check = (
    signal_df
    .groupby("search_volume_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("gsc_impressions", "median"),
        mean_impressions=("gsc_impressions", "mean"),
        median_clicks=("gsc_clicks", "median"),
        total_impressions=("gsc_impressions", "sum")
    )
    .reset_index()
)

print(volume_check)

  search_volume_bucket       n  median_impressions  mean_impressions  \
0        No demand (0)  163631                 1.0        910.821289   
1           Low (1-20)  127670                35.0       1048.938022   
2    Moderate (21-110)   50948                54.0       1312.431544   
3       High (111-390)   17837                42.0       1187.630476   
4     Very high (391+)   16898                23.0        991.765052   

   median_clicks  total_impressions  
0            0.0        115183371.0  
1            0.0         90612511.0  
2            0.0         47814506.0  
3            0.0         14909513.0  
4            0.0         10114020.0  


Verdict: MIXED <br>
Search volume is associated with visibility at the lower end: median impressions increase from 1 for no-demand content to 54 for moderate-demand content. However, the relationship is not monotonic because high- and very-high-demand groups have lower median impressions (42 and 23). Therefore, search volume is a useful context signal but should not be treated as a standalone performance ranking rule.

###Rule:

I use a simple 0–6 baseline action score based on two observable signals: content staleness and search demand. Staleness contributes 0–3 points based on content age, and search demand contributes 0–3 points based on search volume. Higher scores prioritize pages that are both older and associated with greater search demand. The score is a prioritization baseline, not a prediction or a ground-truth label.

Score bands:
- 5–6: REFRESH
- 3–4: IMPROVE
- 1–2: MONITOR
- 0: LOW_PRIORITY

Reason codes:
- STALE_HIGH_DEMAND — older content with meaningful search demand.
- HIGH_DEMAND — meaningful search demand without strong staleness.
- STALE_LOW_DEMAND — older content with little search demand.
- LOW_DEMAND — little or no search demand and limited apparent opportunity.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
import os
import pandas as pd

# Work on the full content inventory
baseline_df = dim_content[
    [
        "content_hash_id",
        "content_age_days",
        "search_volume"
    ]
].copy()

# Missing search volume is treated as no recorded demand
baseline_df["search_volume"] = baseline_df["search_volume"].fillna(0)

# -----------------------------
# 1. Staleness points
# -----------------------------
baseline_df["staleness_points"] = pd.cut(
    baseline_df["content_age_days"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=[0, 1, 2, 3]
).astype(int)

# -----------------------------
# 2. Search-demand points
# -----------------------------
baseline_df["demand_points"] = pd.cut(
    baseline_df["search_volume"],
    bins=[-1, 0, 20, 110, float("inf")],
    labels=[0, 1, 2, 3]
).astype(int)

# -----------------------------
# 3. Total action score
# -----------------------------
baseline_df["action_score"] = (
    baseline_df["staleness_points"]
    + baseline_df["demand_points"]
)

# -----------------------------
# 4. Reason code
# -----------------------------
def get_reason(row):
    if row["staleness_points"] >= 2 and row["demand_points"] >= 2:
        return "STALE_HIGH_DEMAND"
    elif row["demand_points"] >= 2:
        return "HIGH_DEMAND"
    elif row["staleness_points"] >= 2:
        return "STALE_LOW_DEMAND"
    else:
        return "LOW_DEMAND"

baseline_df["reason_code"] = baseline_df.apply(get_reason, axis=1)

# -----------------------------
# 5. Action label
# -----------------------------
def get_action(score):
    if score >= 5:
        return "REFRESH"
    elif score >= 3:
        return "IMPROVE"
    elif score >= 1:
        return "MONITOR"
    else:
        return "LOW_PRIORITY"

baseline_df["action"] = baseline_df["action_score"].apply(get_action)

# -----------------------------
# 6. Rank everything
# -----------------------------
baseline_df = baseline_df.sort_values(
    ["action_score", "content_hash_id"],
    ascending=[False, True]
).reset_index(drop=True)

baseline_df["rank"] = baseline_df.index + 1

# -----------------------------
# 7. Final ranked queue
# -----------------------------
baseline_queue = baseline_df[
    [
        "rank",
        "content_hash_id",
        "action_score",
        "reason_code",
        "action"
    ]
].copy()

print("Ranked queue shape:", baseline_queue.shape)
print("\nTop 10:")
print(baseline_queue.head(10))

Ranked queue shape: (519606, 5)

Top 10:
   rank           content_hash_id  action_score        reason_code   action
0     1  content_0019213f6e9ae2a6             6  STALE_HIGH_DEMAND  REFRESH
1     2  content_0029f2d701f96b83             6  STALE_HIGH_DEMAND  REFRESH
2     3  content_006fbb1dd2fb8a27             6  STALE_HIGH_DEMAND  REFRESH
3     4  content_007aeaa7d49fea99             6  STALE_HIGH_DEMAND  REFRESH
4     5  content_008059cbc87b58b8             6  STALE_HIGH_DEMAND  REFRESH
5     6  content_00acf74dcd049478             6  STALE_HIGH_DEMAND  REFRESH
6     7  content_00bee1fba5dd65fb             6  STALE_HIGH_DEMAND  REFRESH
7     8  content_00f3cc8043b9e19d             6  STALE_HIGH_DEMAND  REFRESH
8     9  content_00fa5ea07b5d9864             6  STALE_HIGH_DEMAND  REFRESH
9    10  content_01037f9acd8202e7             6  STALE_HIGH_DEMAND  REFRESH


In [16]:
# Save the ranked baseline queue
output_path = "work/outputs/baseline_action_score.csv"

os.makedirs("work/outputs", exist_ok=True)

baseline_queue.to_csv(
    output_path,
    index=False
)

print(f"Saved baseline queue to: {output_path}")
print(f"Rows saved: {len(baseline_queue):,}")

Saved baseline queue to: work/outputs/baseline_action_score.csv
Rows saved: 519,606


In [17]:
print(pd.read_csv(output_path).head(10))

   rank           content_hash_id  action_score        reason_code   action
0     1  content_0019213f6e9ae2a6             6  STALE_HIGH_DEMAND  REFRESH
1     2  content_0029f2d701f96b83             6  STALE_HIGH_DEMAND  REFRESH
2     3  content_006fbb1dd2fb8a27             6  STALE_HIGH_DEMAND  REFRESH
3     4  content_007aeaa7d49fea99             6  STALE_HIGH_DEMAND  REFRESH
4     5  content_008059cbc87b58b8             6  STALE_HIGH_DEMAND  REFRESH
5     6  content_00acf74dcd049478             6  STALE_HIGH_DEMAND  REFRESH
6     7  content_00bee1fba5dd65fb             6  STALE_HIGH_DEMAND  REFRESH
7     8  content_00f3cc8043b9e19d             6  STALE_HIGH_DEMAND  REFRESH
8     9  content_00fa5ea07b5d9864             6  STALE_HIGH_DEMAND  REFRESH
9    10  content_01037f9acd8202e7             6  STALE_HIGH_DEMAND  REFRESH


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

The baseline ranked these pages highest because they received the maximum action score of 6 from the combination of high search demand and high content age.

| Rank | Action | Why it's there | What would make it wrong |
|---:|---|---|---|
| 1 | REFRESH | 273 days old with search volume of 320, giving the maximum score. | The page is deleted and unpublished, so refreshing it may be inappropriate. |
| 2 | REFRESH | 228 days old with search volume of 1,000, giving the maximum score. | Missing content-quality information, such as word count, could change the decision. |
| 3 | REFRESH | 276 days old with search volume of 260, giving the maximum score. | The page is deleted and unpublished, so it may no longer be an active refresh candidate. |
| 4 | REFRESH | 339 days old with search volume of 170, giving the maximum score. | The page is deleted and unpublished, and its navigational intent may require a different action. |
| 5 | REFRESH | 301 days old with search volume of 1,300, giving the maximum score. | The page is deleted and unpublished, so high search demand alone does not mean it should be refreshed. |
| 6 | REFRESH | 274 days old with search volume of 170, giving the maximum score. | The page is deleted and unpublished, making the refresh recommendation questionable. |
| 7 | REFRESH | 328 days old with search volume of 480, giving the maximum score. | The page is deleted and unpublished, so the baseline lacks lifecycle-status information. |
| 8 | REFRESH | 328 days old with search volume of 320, giving the maximum score. | The page is deleted and unpublished, so refreshing it may not be appropriate. |
| 9 | REFRESH | 299 days old with search volume of 170, giving the maximum score. | The page is deleted and unpublished, so the recommendation may be invalid. |
| 10 | REFRESH | 302 days old with search volume of 1,000, giving the maximum score. | The page is deleted and unpublished, so the rule may be prioritizing inactive content. |

### Skeptical finding

The top-10 review exposed an important limitation of the baseline. Nine of the ten highest-ranked pages are marked as deleted and unpublished, yet the rule recommends `REFRESH` for all of them. This happens because the score only uses content age and search volume and does not consider content lifecycle status.

Therefore, the baseline should be treated as a prioritization heuristic rather than a complete production action system. A production version would need to account for whether content is currently published and active before recommending a refresh.

In [20]:
# Recreate the top-10 review table

top10_ids = baseline_queue.head(10)["content_hash_id"]

top10_review = dim_content[
    dim_content["content_hash_id"].isin(top10_ids)
][
    [
        "content_hash_id",
        "content_age_days",
        "search_volume",
        "content_type",
        "main_intent",
        "word_count",
        "is_published",
        "is_deleted"
    ]
].copy()

top10_review = baseline_queue.head(10).merge(
    top10_review,
    on="content_hash_id",
    how="left"
)

print(top10_review.to_string(index=False))

 rank          content_hash_id  action_score       reason_code  action  content_age_days  search_volume    content_type   main_intent  word_count  is_published  is_deleted
    1 content_0019213f6e9ae2a6             6 STALE_HIGH_DEMAND REFRESH               273          320.0 keyword article informational      3323.0         False        True
    2 content_0029f2d701f96b83             6 STALE_HIGH_DEMAND REFRESH               228         1000.0 keyword article informational         NaN          True       False
    3 content_006fbb1dd2fb8a27             6 STALE_HIGH_DEMAND REFRESH               276          260.0 keyword article informational      3640.0         False        True
    4 content_007aeaa7d49fea99             6 STALE_HIGH_DEMAND REFRESH               339          170.0 keyword article  navigational      1597.0         False        True
    5 content_008059cbc87b58b8             6 STALE_HIGH_DEMAND REFRESH               301         1300.0 keyword article informational      1

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [21]:
# Identify weak picks in the top 10

weak_picks = top10_review[
    (top10_review["is_deleted"] == True) |
    (top10_review["is_published"] == False)
][
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "content_age_days",
        "search_volume",
        "is_published",
        "is_deleted"
    ]
]

print("WEAK PICKS")
print(weak_picks.to_string(index=False))

print("\nNumber of weak picks:", len(weak_picks))

WEAK PICKS
 rank          content_hash_id  action       reason_code  content_age_days  search_volume  is_published  is_deleted
    1 content_0019213f6e9ae2a6 REFRESH STALE_HIGH_DEMAND               273          320.0         False        True
    3 content_006fbb1dd2fb8a27 REFRESH STALE_HIGH_DEMAND               276          260.0         False        True
    4 content_007aeaa7d49fea99 REFRESH STALE_HIGH_DEMAND               339          170.0         False        True
    5 content_008059cbc87b58b8 REFRESH STALE_HIGH_DEMAND               301         1300.0         False        True
    6 content_00acf74dcd049478 REFRESH STALE_HIGH_DEMAND               274          170.0         False        True
    7 content_00bee1fba5dd65fb REFRESH STALE_HIGH_DEMAND               328          480.0         False        True
    8 content_00f3cc8043b9e19d REFRESH STALE_HIGH_DEMAND               328          320.0         False        True
    9 content_00fa5ea07b5d9864 REFRESH STALE_HIGH_DEMAND     

### Weak picks and leakage finding

The baseline produced 9 weak picks among the top 10 recommendations. All 9 are marked as deleted and unpublished, even though the rule recommends `REFRESH`. This shows that the baseline prioritizes content age and search demand without considering whether the content is currently active.

The leakage check confirms that the baseline score uses only `content_age_days` and `search_volume`. No product/action flags or future performance metrics from `fact_daily` were used to calculate the score.

This is an important limitation of the baseline: it identifies apparent refresh opportunities but does not verify whether the content is currently eligible for action.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.